### Transform Payments Data
- Extract Date and Time from the payment_timestamp and create new columns payment_date and payement_time
- Map payment_status to contain descriptive values (1-Success, 2-Pending, 3- Cancelled, 4- Failed)
- Write transformed data into silver layer

#### 1. Extract Date and Time from the payment_timestamp and create new columns payment_date and payement_time
https://docs.databricks.com/aws/en/sql/language-manual/functions/date_format


In [0]:
import pyspark.sql.functions as F

df = spark.read.table("gizmobox.bronze.py_payments")
extract_df = df.select("payment_id"
                       ,"order_id"
                       ,"payement_status"
                       ,"payment_method"
                       ,F.date_format(df.payment_timstamp, "yyyy-MM-dd").cast("date").alias("payment_date")
                       ,F.date_format(df.payment_timstamp, "HH:mm:ss").alias("payment_time")
                       )

display(extract_df)

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("gizmobox.bronze.py_payments")
extract_df = df.withColumn("payment_date", df.payment_timstamp.cast("date")).withColumn("payment_time", F.date_format(df.payment_timstamp, "HH:mm:ss")).drop("payment_timstamp")
display(extract_df)

### https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions

In [0]:
%sql
SELECT
    *,
    CAST(date_format(payment_timstamp, 'yyyy-MM-dd') AS DATE) AS payement_date,
    date_format(payment_timstamp, 'HH:mm:ss') AS payement_time
FROM gizmobox.bronze.payments
ORDER BY payment_id;

#### 2. Map payment_status to contain descriptive values (1-Success, 2-Pending, 3- Cancelled, 4- Failed)


In [0]:
from pyspark.sql import functions as F

final_df = extract_df.select(
                        "payment_id",
                        "order_id",
                        "payment_date",
                        "payment_time",
                        F.when(extract_df.payement_status == 1, "Success")
                         .when(extract_df.payement_status == 2, "Pending")
                         .when(extract_df.payement_status == 3, "Failed")
                         .when(extract_df.payement_status == 4, "Cancelled")
                         .alias("payment_status"),
                        "payment_method"
                       )
display(final_df)

#### 3. Write transformed data into silver layer

In [0]:
final_df.writeTo("gizmobox.silver.py_payments").createOrReplace()


In [0]:
%sql
SELECT * FROM gizmobox.silver.payments;